# Launch Thermal DIP GPU Backend in Google Colab
### Unsupervised Thermal Image Restoration (Deep Image Prior)

This notebook runs the **FastAPI Backend and PyTorch U-Net DIP worker** on Google Colab's free **NVIDIA Tesla T4 GPU**.

**Instructions:**
1. Go to menu: **Runtime > Change runtime type > select T4 GPU**.
2. Click **Runtime > Run all**.
3. Copy the `PUBLIC BACKEND URL` printed in the last cell and paste it into your local Web Studio (`http://localhost:8000/restoration-studio.html`).

In [ ]:
# 1. Verify CUDA GPU Acceleration
import torch
cuda_ok = torch.cuda.is_available()
print('CUDA Available:', cuda_ok)
if cuda_ok:
    print('GPU Model:', torch.cuda.get_device_name(0))
else:
    print('WARNING: Running on CPU. Please switch runtime: Runtime > Change runtime type > T4 GPU.')

In [ ]:
# 2. Clone Repository and Install Dependencies
from pathlib import Path
import os

if not Path('ml/dip_core').exists() and not Path('DIP_MP/ml/dip_core').exists():
    !git clone https://github.com/likhith-naik01/DIP_MP.git
    %cd DIP_MP
elif Path('DIP_MP').exists():
    %cd DIP_MP
    !git pull origin main

!pip install -q -e ml/
!pip install -q fastapi uvicorn python-multipart pydantic scikit-image

In [ ]:
# 3. Install Cloudflared Tunnel (No tokens or registration required)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
# 4. Launch FastAPI Server & Cloudflare Tunnel
import os, sys, time, re, subprocess
from pathlib import Path

os.environ['PYTHONPATH'] = f'{os.getcwd()}:{os.getcwd()}/ml:{os.getcwd()}/backend'

print('Starting FastAPI GPU server on port 8000...')
server_log = open('server.log', 'w')
server_proc = subprocess.Popen(
    ['uvicorn', 'backend.app.main:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=server_log, stderr=server_log
)
time.sleep(3)

print('Starting Cloudflare tunnel...')
tunnel_log = open('tunnel.log', 'w')
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
    stdout=tunnel_log, stderr=tunnel_log
)

public_url = None
for _ in range(25):
    time.sleep(1)
    if Path('tunnel.log').exists():
        with open('tunnel.log', 'r') as f:
            for line in f:
                match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
                if match:
                    public_url = match.group(0)
                    break
    if public_url:
        break

print('\n' + '='*65)
if public_url:
    print('SUCCESS! COLAB GPU BACKEND IS RUNNING WITH CUDA!')
    print(f'PUBLIC BACKEND URL: {public_url}')
    print('\nCopy this URL and paste it into the Web Studio (Connect GPU).')
    print(f'API Health: {public_url}/api/health')
else:
    print('Tunnel starting. Check tunnel.log:')
    if Path('tunnel.log').exists():
        with open('tunnel.log') as f:
            print(f.read())
print('='*65)

# Stream backend logs in real time
try:
    while True:
        time.sleep(2)
except KeyboardInterrupt:
    print('Stopping servers...')
    server_proc.terminate()
    tunnel_proc.terminate()